# Optional model smoke test

This notebook loads `bert-base-uncased` into the custom capitalization-aware model and runs one forward pass. Skip this while iterating only on tokenization; it downloads and instantiates the full BERT checkpoint.


In [ ]:
from pathlib import Path
import os

COLAB_REPO = Path("/content/drive/MyDrive/Github/CapitalizationEmbeddings")
try:
    from google.colab import drive

    if not COLAB_REPO.exists():
        drive.mount("/content/drive")
except Exception:
    pass

if COLAB_REPO.exists():
    os.chdir(COLAB_REPO)

print("repo:", Path.cwd())
%pip install -q -e . -r requirements-colab.txt

from capitalization_embeddings import configure_huggingface_cache
HF_CACHE_DIR = configure_huggingface_cache()
print("HF cache:", HF_CACHE_DIR)


In [ ]:
from pathlib import Path

if not Path("pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the CapitalizationEmbeddings repo root.")


In [ ]:
import torch
from transformers import AutoTokenizer

from capitalization_embeddings import (
    CapitalizedBertForMaskedLM,
    DataCollatorForCapitalizedLanguageModeling,
    tokenize_with_capitalization,
)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)
examples = [
    "Tom met tom and TOM near iPhone HQ.",
    "NASA hired Alice in New York.",
]
features = [
    tokenize_with_capitalization(
        tokenizer,
        example,
        truncation=True,
        max_length=32,
    )
    for example in examples
]

collator = DataCollatorForCapitalizedLanguageModeling(
    tokenizer=tokenizer,
    mlm_probability=0.30,
)

torch.manual_seed(0)
mlm_batch = collator(features)

model = CapitalizedBertForMaskedLM.from_uncased_pretrained("bert-base-uncased")
model.eval()

with torch.no_grad():
    outputs = model(**mlm_batch)

print("loss:", float(outputs.loss))
print("token logits:", tuple(outputs.logits.shape))
print("capitalization logits:", tuple(outputs.capitalization_logits.shape))
